## Load relevant notebooks

In [0]:
%run "./05_1_orders_integrated"

DataFrame[order_id: string, customer_id: string, order_status: string, order_business_status: string, item_count: bigint, items_price_total: double, shipping_total: double, has_items: boolean, number_of_payments: bigint, total_payment_value: double, payment_types_used: array<string>, has_payment: boolean, is_paid: boolean, payment_matches_items: boolean, expected_order_value: double, order_value_gap: double, order_purchase_timestamp: timestamp, order_approved_at: timestamp, order_delivered_carrier_date: timestamp, order_delivered_customer_date: timestamp, order_estimated_delivery_date: timestamp, is_delivered: boolean, is_completed: boolean]

In [0]:
# Assign the integrated orders DataFrame from previous notebook run
# This allows for downstream gold-layer modeling
# df_orders_integrated is used as the main orders fact source.
df_orders_integrated = orders_integrated

In [0]:
%run "./05_2_customer_semantics"

DataFrame[customer_id: string, customer_unique_id: string, customer_state: string, order_id: string, order_status: string, total_orders: bigint, first_order_ts: timestamp, last_order_ts: timestamp, customer_type: string]

In [0]:
# Assign the semantic customer DataFrame from previous notebook run
# df_customer_semantic will be used to enrich dimensional modeling
df_customer_semantic = customer_semantic

# Dimensional Modeling — Gold Layer Design

## 🇦 Dimension Tables

### 1️⃣ dim_dates

In [0]:
# Display a sample of the integrated orders DataFrame for quick validation
# Enables inspection of business metrics and event timestamp coverage
display(df_orders_integrated.limit(5))

order_id,customer_id,order_status,order_business_status,item_count,items_price_total,shipping_total,has_items,number_of_payments,total_payment_value,payment_types_used,has_payment,is_paid,payment_matches_items,expected_order_value,order_value_gap,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,is_completed
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,completed,1,29.99,8.72,true,3,38.71,"List(credit_card, voucher)",true,true,true,38.71,0.0,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z,true,true
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,completed,1,118.7,22.76,true,1,141.46,List(boleto),true,true,true,141.46,0.0,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z,true,true
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,completed,1,159.9,19.22,true,1,179.12,List(credit_card),true,true,true,179.12,0.0,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z,true,true
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,completed,1,45.0,27.2,true,1,72.2,List(credit_card),true,true,true,72.2,0.0,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z,true,true
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,completed,1,19.9,8.72,true,1,28.62,List(credit_card),true,true,true,28.62,0.0,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z,true,true


In [0]:
from pyspark.sql.functions import col
from pyspark.sql.functions import min, max

# Unpivot (stack) key event timestamps from orders into a single column
# Used to compute global min/max dates for dimension table generation

stacked_dates = df_orders_integrated.select(
    col("order_purchase_timestamp").alias("event_ts"),
).unionByName(
    df_orders_integrated.select(col("order_delivered_carrier_date").alias("event_ts"))
).unionByName(
    df_orders_integrated.select(col("order_delivered_customer_date").alias("event_ts"))
).unionByName(
    df_orders_integrated.select(col("order_estimated_delivery_date").alias("event_ts"))
)

# Compute global min and max across all event dates
min_max_dates = stacked_dates.agg(
    min("event_ts").alias("min_date"),
    max("event_ts").alias("max_date")
)

display(min_max_dates)

min_date,max_date
2016-09-04T21:15:19.000Z,2018-11-12T00:00:00.000Z


In [0]:
from pyspark.sql.functions import col, sequence, to_date, explode, year, month, dayofmonth, dayofweek, quarter, lit, date_format
from pyspark.sql.types import IntegerType
from pyspark.sql import functions as F
from pyspark.sql.functions import col, substring, initcap

# Create a date dimension (dim_dates) spanning the min/max event dates
# Each date gets business, calendar, and sortable attributes

df_dates = min_max_dates.select(
    explode(sequence(to_date(col("min_date")), to_date(col("max_date")))).alias("date")
)

dim_dates = df_dates \
.withColumn("date_id", date_format(col("date"), "yyyyMMdd").cast(IntegerType())) \
.withColumn("year", year(col("date"))) \
.withColumn("month_number", month(col("date")))\
.withColumn("month_name", date_format(col("date"), "MMMM"))\
.withColumn("day_of_month", dayofmonth(col("date"))) \
.withColumn("day_of_week", date_format(col("date"), "EEEE")) \
.withColumn("quarter", quarter(col("date"))) \
.withColumn("isweekend", (dayofweek(col("date")).isin([1,7])).cast("boolean")) \
.withColumn("month_year", F.date_format("date", "MMM yyyy")) \
.withColumn("month_year_sort", F.year("date") * 100 + F.month("date")) \
.withColumn("month_name_short", substring(initcap(col("month_name")), 1, 3))

display(dim_dates.limit(5))

date,date_id,year,month_number,month_name,day_of_month,day_of_week,quarter,isweekend,month_year,month_year_sort,month_name_short
2016-09-04,20160904,2016,9,September,4,Sunday,3,true,Sep 2016,201609,Sep
2016-09-05,20160905,2016,9,September,5,Monday,3,false,Sep 2016,201609,Sep
2016-09-06,20160906,2016,9,September,6,Tuesday,3,false,Sep 2016,201609,Sep
2016-09-07,20160907,2016,9,September,7,Wednesday,3,false,Sep 2016,201609,Sep
2016-09-08,20160908,2016,9,September,8,Thursday,3,false,Sep 2016,201609,Sep


In [0]:
# Order dates for easier downstream query and reporting
# Write date dimension to gold layer, overwriting if exists
# Show first 5 rows as sanity check

dim_dates = dim_dates.orderBy(col("month_number").asc())
dim_dates.write.mode("overwrite").saveAsTable("retail_catalog.gold.dim_dates")
display(dim_dates.limit(5))

date,date_id,year,month_number,month_name,day_of_month,day_of_week,quarter,isweekend,month_year,month_year_sort,month_name_short
2017-01-03,20170103,2017,1,January,3,Tuesday,1,false,Jan 2017,201701,Jan
2017-01-02,20170102,2017,1,January,2,Monday,1,false,Jan 2017,201701,Jan
2017-01-01,20170101,2017,1,January,1,Sunday,1,true,Jan 2017,201701,Jan
2017-01-04,20170104,2017,1,January,4,Wednesday,1,false,Jan 2017,201701,Jan
2017-01-05,20170105,2017,1,January,5,Thursday,1,false,Jan 2017,201701,Jan


### 2️⃣ dim_customers

In [0]:
# Load customers silver-layer DataFrame for customer dimension modeling
customers = spark.table("retail_catalog.silver.customers")

In [0]:
# Alias raw and semantic customer tables for join
customers_aliased = customers.alias("c")
customer_semantic_aliased = df_customer_semantic.alias("s")
# Build gold-layer dim_customers table by joining and selecting enriched fields
dim_customers = customers_aliased.join(customer_semantic_aliased, on="customer_id", how="inner")
dim_customers = dim_customers.select("customer_id","s.customer_unique_id","s.customer_state", "c.customer_city",   "c.customer_zip_code_prefix", "total_orders", "first_order_ts", "last_order_ts", "customer_type")

display(dim_customers.limit(5))

customer_id,customer_unique_id,customer_state,customer_city,customer_zip_code_prefix,total_orders,first_order_ts,last_order_ts,customer_type
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,SP,franca,14409,1,2017-05-16T15:05:35.000Z,2017-05-16T15:05:35.000Z,NEW
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,SP,sao bernardo do campo,9790,1,2018-01-12T20:48:24.000Z,2018-01-12T20:48:24.000Z,NEW
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,SP,sao paulo,1151,1,2018-05-19T16:07:45.000Z,2018-05-19T16:07:45.000Z,NEW
b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,SP,mogi das cruzes,8775,1,2018-03-13T16:06:38.000Z,2018-03-13T16:06:38.000Z,NEW
4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,SP,campinas,13056,1,2018-07-29T09:51:30.000Z,2018-07-29T09:51:30.000Z,NEW


In [0]:
# Save the customers dimension table to gold layer for downstream analytics and reporting
dim_customers.write.mode("overwrite").saveAsTable("retail_catalog.gold.dim_customers")

### 3️⃣ dim_products

In [0]:
%run "./05_3_product_category_reference_table"

DataFrame[product_id: string, product_weight_g: int, product_length_cm: int, product_height_cm: int, product_width_cm: int, product_category_name: string, product_category_name_english: string, grouped_category: string]

In [0]:
# Assign products reference table from prior notebook run and select gold-layer dimensional columns
df_products = products
df_products = df_products.select("product_id","product_weight_g","product_length_cm","product_height_cm","product_width_cm","product_category_name","product_category_name_english","grouped_category")

In [0]:
# Remove duplicate products (by product_id) from dim_products
# Display sample for sanity check
dim_products = df_products.dropDuplicates(["product_id"])
display(dim_products.limit(5))

product_id,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name,product_category_name_english,grouped_category
e1d1d22e9f8122a4ec1533b032c12562,2150,70,8,34,ferramentas_jardim,garden_tools,Home & Furniture
ce5b91848b91118daffb3af53b747475,1388,34,9,31,esporte_lazer,sports_leisure,Sports & Leisure
1c6fb703c624b381a20f21f757694866,725,22,17,15,brinquedos,toys,Kids & Baby
4e04ffb7dd3739ecfc37de8927dd586c,500,24,7,16,papelaria,stationery,Other / Business & Misc
0992c6cba95a13bfa68ea7d5e22d478b,3850,29,12,24,ferramentas_jardim,garden_tools,Home & Furniture


In [0]:
# Write products dimension table to gold layer for BI use
dim_products.write.mode("overwrite").saveAsTable("retail_catalog.gold.dim_products")

## 🇧 Fact Tables

### 1️⃣ fact_orders

In [0]:
# Join orders and customer semantic tables for fact_orders
# Alias for clarity, inner join ensures only valid orders with customer context

orders_integrated_aliased = df_orders_integrated.alias("ord")
customer_semantic_aliased = df_customer_semantic.alias("cust")
ord_cust_joined = orders_integrated_aliased.join(customer_semantic_aliased, on="order_id", how="inner")
display(ord_cust_joined.limit(5))

order_id,customer_id,order_status,order_business_status,item_count,items_price_total,shipping_total,has_items,number_of_payments,total_payment_value,payment_types_used,has_payment,is_paid,payment_matches_items,expected_order_value,order_value_gap,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,is_completed,customer_id,customer_unique_id,customer_state,order_status,total_orders,first_order_ts,last_order_ts,customer_type
66e4624ae69e7dc89bd50222b59f581f,684fa6da5134b9e4dab731e00011712d,delivered,completed,2,45.98,45.7,true,1,91.68,List(credit_card),true,true,true,91.68,0.0,2018-03-09T14:50:15.000Z,2018-03-09T15:40:39.000Z,2018-03-15T00:31:19.000Z,2018-04-03T13:28:46.000Z,2018-04-02T00:00:00.000Z,true,true,684fa6da5134b9e4dab731e00011712d,ddf60e20e6e262e2136801ce5cd628b0,SE,delivered,1,2018-03-09T14:50:15.000Z,2018-03-09T14:50:15.000Z,NEW
abc5ec9ecaec740b498a37f19c29a8c0,85aa7dc7ea24c96b5ac7864f13922495,delivered,completed,1,119.9,21.16,true,1,141.06,List(credit_card),true,true,true,141.06,0.0,2017-04-30T22:43:56.000Z,2017-04-30T22:55:09.000Z,2017-05-02T13:58:16.000Z,2017-05-12T12:04:38.000Z,2017-06-05T00:00:00.000Z,true,true,85aa7dc7ea24c96b5ac7864f13922495,22be41ad7d58c7774c6e036348a042d0,RS,delivered,1,2017-04-30T22:43:56.000Z,2017-04-30T22:43:56.000Z,NEW
638f79643fd5681c6e6c6ef1f1ab700d,69d8d9775287ba592a957eba2211d351,delivered,completed,1,178.99,19.13,true,1,198.12,List(credit_card),true,true,true,198.12,0.0,2018-04-10T14:58:16.000Z,2018-04-10T15:11:08.000Z,2018-04-17T13:16:29.000Z,2018-04-27T23:17:26.000Z,2018-05-11T00:00:00.000Z,true,true,69d8d9775287ba592a957eba2211d351,4e1e21d422058c75922a29548f43479d,RJ,delivered,1,2018-04-10T14:58:16.000Z,2018-04-10T14:58:16.000Z,NEW
879195f2460f97ab2f3bf119f2e8638b,09cdb569d94234872c468f2711ca0f91,delivered,completed,1,27.9,11.85,true,1,39.75,List(credit_card),true,true,true,39.75,0.0,2018-02-03T14:56:55.000Z,2018-02-03T15:11:02.000Z,2018-02-05T15:54:50.000Z,2018-02-23T22:59:22.000Z,2018-03-01T00:00:00.000Z,true,true,09cdb569d94234872c468f2711ca0f91,ae3759d2ee307574752179acccd595b6,SP,delivered,1,2018-02-03T14:56:55.000Z,2018-02-03T14:56:55.000Z,NEW
f373335aac9a659de916f7170b8bc07a,f06a94a401e52fb019c72f2e8bbf6a2f,shipped,paid_not_delivered,1,35.9,23.28,true,1,59.18,List(credit_card),true,true,true,59.18,0.0,2018-03-17T15:32:31.000Z,2018-03-17T15:48:40.000Z,2018-03-20T21:08:28.000Z,null,2018-04-13T00:00:00.000Z,false,false,f06a94a401e52fb019c72f2e8bbf6a2f,d4b1297d645ec19df4cd0af8cd6fe14a,BA,shipped,1,2018-03-17T15:32:31.000Z,2018-03-17T15:32:31.000Z,NEW


In [0]:
from pyspark.sql import functions as F

# Add surrogate key (date_id) for time-based analysis by joining dim_dates
# Drop redundant columns for final output
ord_cust_joined = ord_cust_joined.withColumn(
    "order_date",
    F.to_date("order_purchase_timestamp")
)
ord_cust_joined = (
    ord_cust_joined
    .join(
        dim_dates.select("date", "date_id"),
        ord_cust_joined.order_date == dim_dates.date,
        "left"
    )
)
ord_cust_joined_final = (
    ord_cust_joined
    .drop("order_date", "date")
)

In [0]:
# Count how many records in ord_cust_joined_final are missing a date_id (date dimension surrogate key)
# This helps identify orders without a matching date for auditing and fixing QA issues
ord_cust_joined_final.filter(
    F.col("date_id").isNull()
).count()

0

In [0]:
# Display the actual offending records missing date_id, to debug/fix pipeline issues
# Useful for tracking problematic input data or join logic on dates
display(ord_cust_joined_final.filter(
    F.col("date_id").isNull()))

order_id,customer_id,order_status,order_business_status,item_count,items_price_total,shipping_total,has_items,number_of_payments,total_payment_value,payment_types_used,has_payment,is_paid,payment_matches_items,expected_order_value,order_value_gap,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,is_completed,customer_id,customer_unique_id,customer_state,order_status,total_orders,first_order_ts,last_order_ts,customer_type,date_id


In [0]:
# Show a sample of the final star-joined orders and customer features for fact_orders modeling
# Helpful for verification before persisting or further normalization
display(ord_cust_joined_final.limit(5))

order_id,customer_id,order_status,order_business_status,item_count,items_price_total,shipping_total,has_items,number_of_payments,total_payment_value,payment_types_used,has_payment,is_paid,payment_matches_items,expected_order_value,order_value_gap,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,is_delivered,is_completed,customer_id,customer_unique_id,customer_state,order_status,total_orders,first_order_ts,last_order_ts,customer_type,date_id
67b47f35ec923e39f1203f3cfa80f5e3,0dd5a932b1e29daa12845dbf696cbb54,delivered,completed,1,28.9,12.79,true,1,41.69,List(credit_card),true,true,true,41.69,0.0,2018-06-03T18:24:33.000Z,2018-06-03T18:49:57.000Z,2018-06-04T15:11:00.000Z,2018-06-08T18:58:53.000Z,2018-07-11T00:00:00.000Z,true,true,0dd5a932b1e29daa12845dbf696cbb54,76c9a12722537319e44c31e70b7815c3,SP,delivered,2,2017-08-15T22:42:16.000Z,2018-06-03T18:24:33.000Z,RETURNING,20180603
b7ed5b357d48b6c8f588c5cf0d7ddf5d,92148a85c89c0da621cdc0e1810a23f7,delivered,completed,1,149.9,23.63,true,1,173.53,List(credit_card),true,true,true,173.53,0.0,2018-03-10T23:24:50.000Z,2018-03-11T00:00:30.000Z,2018-03-12T17:08:26.000Z,2018-04-17T17:37:54.000Z,2018-04-02T00:00:00.000Z,true,true,92148a85c89c0da621cdc0e1810a23f7,5d81094365fbd7cb0e548f4508fa5e62,RJ,delivered,1,2018-03-10T23:24:50.000Z,2018-03-10T23:24:50.000Z,NEW,20180310
476984a30a6929077bedabbefc98ce38,98647ebe3068d5566c1c2cf026d72239,delivered,completed,1,68.0,30.28,true,1,98.28,List(credit_card),true,true,true,98.28,0.0,2018-06-20T16:44:28.000Z,2018-06-20T17:01:07.000Z,2018-06-21T09:14:00.000Z,2018-06-26T18:58:22.000Z,2018-07-19T00:00:00.000Z,true,true,98647ebe3068d5566c1c2cf026d72239,aa75aabc388e5d825562397365e0ac61,SP,delivered,1,2018-06-20T16:44:28.000Z,2018-06-20T16:44:28.000Z,NEW,20180620
c788aeedf593229d4ee84e790c2e5483,838aa327b0491d3cb3e9c1c6d9bb382c,delivered,completed,1,89.9,27.1,true,1,117.0,List(credit_card),true,true,true,117.0,0.0,2017-03-05T22:44:58.000Z,2017-03-05T22:55:40.000Z,2017-03-06T08:48:21.000Z,2017-03-20T07:23:36.000Z,2017-04-04T00:00:00.000Z,true,true,838aa327b0491d3cb3e9c1c6d9bb382c,116490c868c239247db4417b68c9b435,BA,delivered,1,2017-03-05T22:44:58.000Z,2017-03-05T22:44:58.000Z,NEW,20170305
00bdcdda88e6b02977fc6ce3d412c600,45ba03e2c6bbb5dc48131ba32ec3ae5e,delivered,completed,1,118.9,18.93,true,1,137.83,List(credit_card),true,true,true,137.83,0.0,2018-06-12T09:51:55.000Z,2018-06-12T10:21:14.000Z,2018-06-12T14:10:00.000Z,2018-06-18T18:20:48.000Z,2018-07-24T00:00:00.000Z,true,true,45ba03e2c6bbb5dc48131ba32ec3ae5e,1148e7e1daf6d2378b814e90506e9d26,RJ,delivered,1,2018-06-12T09:51:55.000Z,2018-06-12T09:51:55.000Z,NEW,20180612


In [0]:
from pyspark.sql import functions as F

# Build the fact_orders table from the star-joined results
# Select key, conformed, and normalized business columns for reporting/BI
fact_orders = ord_cust_joined.select(
    # Keys
    "ord.order_id", #
    "ord.customer_id", #
    "date_id", #
    
    # order Flags / Status
    "ord.order_status", #
    "order_business_status", #
    "is_delivered", #
    "is_completed", #
    "has_items", #
    "has_payment", #
    "is_paid", #
    # "payment_matches_items",
    
    # Financial Metrics
    "number_of_payments", #
    "total_payment_value", #
    "payment_types_used", #
    "item_count", #
    # "items_price_total",
    # "shipping_total",
    # "gross_order_value", #
    # "expected_order_value",
    # "order_value_gap",
    
    # Order Timestamps (Normalized)
    "order_purchase_timestamp", #
    "order_approved_at", #
    "order_delivered_carrier_date", #
    "order_delivered_customer_date", #
    "order_estimated_delivery_date", #
    # "order_created_ts_utc", #
    # "order_delivered_ts_utc", #
    # "order_estimated_delivery_ts_utc" #
)

# Show a sample for final QA before write
display(fact_orders.limit(5))

order_id,customer_id,date_id,order_status,order_business_status,is_delivered,is_completed,has_items,has_payment,is_paid,number_of_payments,total_payment_value,payment_types_used,item_count,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,20171002,delivered,completed,true,true,true,true,true,3,38.71,"List(credit_card, voucher)",1,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,20180724,delivered,completed,true,true,true,true,true,1,141.46,List(boleto),1,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,20180808,delivered,completed,true,true,true,true,true,1,179.12,List(credit_card),1,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,20171118,delivered,completed,true,true,true,true,true,1,72.2,List(credit_card),1,2017-11-18T19:28:06.000Z,2017-11-18T19:45:59.000Z,2017-11-22T13:39:59.000Z,2017-12-02T00:28:42.000Z,2017-12-15T00:00:00.000Z
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,20180213,delivered,completed,true,true,true,true,true,1,28.62,List(credit_card),1,2018-02-13T21:18:39.000Z,2018-02-13T22:20:29.000Z,2018-02-14T19:46:34.000Z,2018-02-16T18:17:02.000Z,2018-02-26T00:00:00.000Z


In [0]:
# Write the fact_orders table to the gold layer for BI/analytics
# Overwrites prior data for a clean, up-to-date star model
fact_orders.write.mode("overwrite").saveAsTable("retail_catalog.gold.fact_orders")

### 2️⃣ fact_order_items

In [0]:
# Load the silver-layer order items table for dimensional/fact modeling
order_items = spark.table("retail_catalog.silver.orderitems")

In [0]:
# Preview sample rows from order items before aggregation or modeling
# Ensures data distribution and schema look correct
display(order_items.limit(5))

order_id,order_item_id,product_id,seller_id,price,shipping_charges
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.9,18.14


In [0]:
from pyspark.sql.functions import count, sum, col, when, collect_set, round
from pyspark.sql import Window

# Aggregate row-level metrics for fact_order_items granularity
# Compute the count of items per order, item value, and value with/without shipping
order_items_agg = order_items.withColumn("item_count", count("*").over(Window.partitionBy("order_id"))) \
                            .withColumn("item_total_value", round(col("price") + col("shipping_charges"), 2))


In [0]:
# Select key metrics and identifiers for the gold-layer fact_order_items table
# Produces one row per product/seller/order for granular analysis
fact_order_items = order_items_agg.select(
    "order_id",
    "product_id",
    "seller_id",
    "price",
    "shipping_charges",
    "item_count",
    "item_total_value"
)

# Preview for QA before writing to gold layer
display(fact_order_items.limit(5))

order_id,product_id,seller_id,price,shipping_charges,item_count,item_total_value
00010242fe8c5a6d1ba2dd792cb16214,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,58.9,13.29,1,72.19
00018f77f2f0320c557190d7a144bdd3,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,239.9,19.93,1,259.83
000229ec398224ef6ca0657da4fc703e,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,199.0,17.87,1,216.87
00024acbcdf0a6daa1e931b038114c75,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,12.99,12.79,1,25.78
00042b26cf59d7ce69dfabb4e55b4fd9,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,199.9,18.14,1,218.04


In [0]:
# Write the fact_order_items table to the gold layer for BI/analytics
# Overwrites prior partitions for incremental ETL or full reloads
fact_order_items.write.mode("overwrite").saveAsTable("retail_catalog.gold.fact_order_items")